# 49 — G-Eval and Custom Metrics

**What you'll learn:**
- `GEval` for defining evaluation criteria in natural language
- `evaluation_steps` to enable chain-of-thought (CoT) scoring
- `BaseMetric` subclassing for deterministic checks
- Combining G-Eval with DeepEval's built-in metrics in one `evaluate()` call

**Context — G-Eval (Liu et al. 2023):**
Standard LLM judges ask for a raw score with no reasoning, leading to inconsistency and position bias.
G-Eval fixes this by requiring the judge to generate reasoning steps *before* assigning a score.
Result: **Spearman correlation with human judgment rises from 0.43 (ROUGE-L) to 0.87 (G-Eval)**.

> Paper: *G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment* — Liu et al., 2023

In [ ]:
# Install dependencies (Colab guard)
import sys

if "google.colab" in sys.modules:
    %pip install -q deepeval langchain langchain-openai python-dotenv

In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API key configured.")

In [ ]:
# ── G-Eval: LLM-as-judge with Chain-of-Thought ────────────────────────────────
#
# Standard LLM judge: "Rate this answer 1-10." → inconsistent, no reasoning
#
# G-Eval approach:
#   1. Define criteria: "Is the answer concise?"
#   2. Define evaluation_steps: explicit CoT reasoning instructions
#   3. Judge generates reasoning FIRST, then assigns score
#
# Why it works better:
#   - Forces systematic evaluation before scoring
#   - Reduces position bias (long answers aren't auto-penalized)
#   - Spearman correlation with humans: G-Eval 0.87 vs ROUGE-L 0.43
#
# When to use GEval vs built-in metrics:
#   GEval:     tone, format compliance, domain accuracy, style, code quality
#   Built-in:  Faithfulness, AnswerRelevancy (standardized + benchmarked)

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

conciseness = GEval(
    name="Conciseness",
    criteria="The response answers the question without unnecessary padding or repetition.",
    evaluation_steps=[
        "Count the number of sentences in the response.",
        "Identify any sentences that repeat information already stated.",
        "Identify filler phrases that add length without meaning.",
        "Score based on how directly the answer addresses the question.",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
    threshold=0.7,
)
print("GEval metric defined:", conciseness.name)

In [ ]:
# Run Conciseness metric on concise vs verbose outputs
# Expected: concise_output scores ~0.9, verbose_output scores ~0.3-0.5

concise_case = LLMTestCase(
    input="Explain what an API is in one sentence.",
    actual_output="An API is an interface that lets software systems communicate with each other.",
)

verbose_case = LLMTestCase(
    input="Explain what an API is in one sentence.",
    actual_output=(
        "Well, an API, which stands for Application Programming Interface, "
        "is essentially a set of rules and protocols that allows different software "
        "applications to communicate and interact with each other, enabling them to "
        "share data and functionality in a standardized way, which is really quite "
        "fundamental to modern software development."
    ),
)

for label, case in [("Concise", concise_case), ("Verbose", verbose_case)]:
    conciseness.measure(case)
    print(f"[{label}] score={conciseness.score:.2f} | passed={conciseness.is_successful()}")
    print(f"  reason: {conciseness.reason}\n")

In [ ]:
import json

from deepeval.metrics.base_metric import BaseMetric

# ── Two more GEval metrics ─────────────────────────────────────────────────────

format_adherence = GEval(
    name="FormatAdherence",
    criteria="The response uses a numbered list format with exactly 3 items.",
    evaluation_steps=[
        "Check if the response contains numbered items (1., 2., 3.).",
        "Count the number of list items.",
        "Verify there are exactly 3 items.",
        "Score 1.0 if format is correct and complete, lower otherwise.",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
    threshold=0.8,
)

accuracy = GEval(
    name="FactualAccuracy",
    criteria="The response contains factually correct information about the topic.",
    evaluation_steps=[
        "Identify all factual claims in the response.",
        "Verify each claim against known facts.",
        "Note any incorrect or misleading statements.",
        "Score based on the proportion of correct facts.",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
    threshold=0.8,
)

# ── Deterministic BaseMetric: JSON schema validation ──────────────────────────


class JsonSchemaMetric(BaseMetric):
    """Checks that output is valid JSON with required field types."""

    def __init__(self, required_schema: dict, threshold: float = 1.0):
        self.required_schema = required_schema  # {field: expected_type}
        self.threshold = threshold
        self.score = 0.0
        self.reason = ""

    @property
    def name(self) -> str:
        return "JsonSchemaMetric"

    def measure(self, test_case: LLMTestCase, *args, **kwargs) -> float:
        output = test_case.actual_output
        try:
            data = json.loads(output)
        except (json.JSONDecodeError, TypeError):
            self.score = 0.0
            self.reason = "Output is not valid JSON"
            return self.score

        errors = []
        for field, expected_type in self.required_schema.items():
            if field not in data:
                errors.append(f"Missing field: {field}")
            elif not isinstance(data[field], expected_type):
                errors.append(
                    f"{field} expected {expected_type.__name__}, got {type(data[field]).__name__}"
                )

        if errors:
            self.score = 0.0
            self.reason = "; ".join(errors)
        else:
            self.score = 1.0
            self.reason = "All schema checks passed"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold


# Run all three
json_validator = JsonSchemaMetric(required_schema={"name": str, "age": int, "role": str})

json_outputs = [
    '{"name": "Alice", "age": 30, "role": "engineer"}',
    '{"name": "Bob", "age": "thirty"}',
    "name: Charlie, age: 25",
]

print("=== JsonSchemaMetric (deterministic) ===")
for output in json_outputs:
    case = LLMTestCase(input="", actual_output=output)
    json_validator.measure(case)
    status = "PASS" if json_validator.is_successful() else "FAIL"
    print(f"  [{status}] {output[:50]!r}")
    print(f"         reason: {json_validator.reason}")

print("\n=== FormatAdherence ===")
for label, output in [
    (
        "numbered list",
        "1. Static typing catches errors at compile time\n2. Better IDE support and autocomplete\n3. Improved code maintainability",
    ),
    (
        "prose",
        "TypeScript has static typing, which is good. Also IDE support is nice. And it helps with maintainability.",
    ),
]:
    tc = LLMTestCase(input="List the top 3 benefits of using TypeScript.", actual_output=output)
    format_adherence.measure(tc)
    print(f"  [{label}] score={format_adherence.score:.2f}")

print("\n=== FactualAccuracy ===")
for label, output in [
    ("correct", "Python was first released in 1991."),
    ("wrong year", "Python was released in 2001."),
]:
    tc = LLMTestCase(input="What year was Python released?", actual_output=output)
    accuracy.measure(tc)
    print(f"  [{label}] score={accuracy.score:.2f}")

In [ ]:
# Combine G-Eval + built-in metric in one evaluate() call
# You can freely mix GEval, BaseMetric subclasses, and DeepEval built-ins.

from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric

answer_relevancy = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini",
)

test_cases = [
    LLMTestCase(
        input="Explain what an API is in one sentence.",
        actual_output="An API is an interface that lets software systems communicate with each other.",
    ),
    LLMTestCase(
        input="Explain what an API is in one sentence.",
        actual_output=(
            "Well, an API, which stands for Application Programming Interface, "
            "is a set of rules that allows software to communicate, which is "
            "quite fundamental to modern software development."
        ),
    ),
]

results = evaluate(
    test_cases=test_cases,
    metrics=[conciseness, answer_relevancy],
    run_async=False,
)

print("\n=== Combined Evaluation Results ===")
print(f"{'Test Case':<10} {'Conciseness':<15} {'AnswerRelevancy':<16} {'Overall'}")
print("-" * 55)
for i, result in enumerate(results.test_results):
    scores = {m.name: m.score for m in result.metrics_data}
    c = scores.get("Conciseness", 0)
    r = scores.get("Answer Relevancy", 0)
    passed = all(m.success for m in result.metrics_data)
    print(f"Case {i + 1:<6} {c:<15.2f} {r:<16.2f} {'PASS' if passed else 'FAIL'}")

## Exercises

1. **Email tone GEval**: Write a `GEval` metric named `EmailTone` with criteria "The response is professional and friendly" and 4 evaluation steps. Test it on a formal email vs a casual one.

2. **Word count BaseMetric**: Write a deterministic `BaseMetric` subclass that checks the answer length is between 50 and 200 words. Return score 1.0 if in range, 0.0 otherwise.

3. **Combined call**: Combine `Conciseness` + `AnswerRelevancy` in one `evaluate()` call on the same test case. Do both metrics agree when an answer is concise but off-topic?

4. **Threshold stress test**: Take the verbose API explanation and evaluate it against `conciseness` with `threshold=0.9`. Which metric fails first when the threshold is strict?

---

**What's next:**
- `47-deepeval-rag-metrics` — Faithfulness, ContextualPrecision, ContextualRecall for RAG pipelines
- `48-deepeval-hallucination-bias` — Hallucination detection and bias metrics
- `50-deepeval-agentic-eval` — Evaluating multi-step agentic workflows with tool call traces